# Chapter 0 – Check Your Setup

Run this notebook first. It checks that your environment has everything the
workshop needs.

**Use Run → Run All Cells** (or **Kernel → Restart Kernel and Run All Cells**).
A tick appears beside each check as soon as it finishes. The first run can
take a minute while Python loads each package for the first time; later runs
take a few seconds.

- ✅ means that part works.
- ❌ means something is wrong. Carry on to the end, then show the red
  message to a helper.

## 1. Set up the checker

In [ ]:
import importlib
import sys
import time
from pathlib import Path
from IPython.display import HTML, display

results = {}  # check name -> passed?


def check(label, func):
    """Run func() and show a tick with its result, or a cross with the error."""
    start = time.perf_counter()
    try:
        detail = func()
        ok = True
    except Exception as exc:
        detail, ok = f"{type(exc).__name__}: {exc}", False
    took = time.perf_counter() - start
    results[label] = ok
    timing = f" <span style='opacity:.5'>({took:.1f} s)</span>" if took >= 0.5 else ""
    style = "opacity:.7" if ok else "color:#c62828"
    display(HTML(
        f"<div style='font-size:1.05em;line-height:1.6'>{'✅' if ok else '❌'} "
        f"<b>{label}</b> <span style='{style}'>"
        f"{'' if detail is None else detail}</span>{timing}</div>"
    ))


def python_version():
    version = sys.version.split()[0]
    if sys.version_info < (3, 10):
        raise RuntimeError(f"Python {version} is too old; need 3.10 or newer")
    return version


check("Python", python_version)

## 2. Python packages

In [ ]:
PACKAGES = [
    ("numpy", "NumPy"), ("scipy", "SciPy"), ("pandas", "pandas"),
    ("xarray", "xarray"), ("rasterio", "rasterio"), ("rioxarray", "rioxarray"),
    ("pyproj", "pyproj"), ("pystac", "pystac"), ("laspy", "laspy"),
    ("lazrs", "lazrs (LAZ decompression)"), ("requests", "requests"),
    ("plotly", "Plotly"), ("folium", "Folium"), ("matplotlib", "Matplotlib"),
    ("ipywidgets", "ipywidgets"), ("anywidget", "anywidget"),
]
for module, name in PACKAGES:
    check(name, lambda m=module: getattr(importlib.import_module(m), "__version__", "installed"))

check("GDAL (inside rasterio)", lambda: importlib.import_module("rasterio").__gdal_version__)

## 3. Workshop files and interactive plots

In [ ]:
def import_utils():
    global wu
    import workshop_utils as wu
    return "found beside this notebook"


def figure_widget():
    import plotly.graph_objects as go
    go.FigureWidget()  # Chapter 3's interactive plots need this
    return "OK"


def outputs_writable():
    out = Path("outputs")
    out.mkdir(exist_ok=True)
    probe = out / ".write_test"
    probe.write_text("ok")
    probe.unlink()
    return str(out.resolve())


check("workshop_utils.py", import_utils)
check("Plotly FigureWidget", figure_widget)
check("Can save to outputs/", outputs_writable)

## 4. Internet: the workshop data

The imagery and LiDAR stream from the workshop STAC catalogue, so these checks
need an internet connection. Each one reads only a tiny piece of the data.

In [ ]:
def stac():
    global item
    import pystac
    item = pystac.Collection.from_file(wu.STAC_COLLECTION_URL).get_item(wu.ITEM_ID)
    if item is None:
        raise KeyError(f"survey {wu.ITEM_ID} is missing from the catalogue")
    return f"survey {item.id}, {len(item.assets)} data files"


def read_imagery():
    import rasterio
    from rasterio.windows import from_bounds
    left, bottom, *_ = wu.AOI_BOUNDS
    with rasterio.open(item.assets["ms"].get_absolute_href()) as src:
        # a 2 m x 2 m patch of the study area
        window = from_bounds(left, bottom, left + 2, bottom + 2, transform=src.transform)
        data = src.read(window=window.round_offsets().round_lengths())
    return f"{data.shape[0]} bands, {data.shape[2]} x {data.shape[1]} px"


def lidar_header():
    from laspy import CopcReader
    with CopcReader.open(wu.get_asset_href(wu.LIDAR_ITEM_ID, asset_key="pointcloud")) as reader:
        return f"{reader.header.point_count:,} points in the full survey"


check("STAC catalogue", stac)
check("Read drone imagery", read_imagery)
check("Open LiDAR point cloud", lidar_header)

## 5. Result

In [ ]:
failed = [name for name, ok in results.items() if not ok]
if failed:
    display(HTML(
        "<div style='padding:10px;border-radius:6px;background:#fdecea;color:#b71c1c;font-size:1.2em'>"
        f"❌ {len(failed)} of {len(results)} checks failed: {', '.join(failed)}. "
        "Show this notebook to a helper.</div>"))
else:
    display(HTML(
        "<div style='padding:10px;border-radius:6px;background:#e8f5e9;color:#1b5e20;font-size:1.2em'>"
        f"🎉 All {len(results)} checks passed. Now check the chart and slider below.</div>"))

## 6. Check that plots and widgets display

Python can't tell whether your browser draws plots and widgets, so check these
by eye. You should see a small chart and a slider, and dragging the slider
should change the text under it.

If the chart or slider is missing, or shows as plain text such as
`IntSlider(value=5…)`, tell a helper.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets

pio.renderers.default = "plotly_mimetype"

x = np.linspace(0, 2 * np.pi, 100)
fig = go.Figure(go.Scatter(x=x, y=np.sin(x), mode="lines"))
fig.update_layout(title="If you can see this wave, plots work ✅", height=300,
                  margin=dict(l=40, r=20, t=50, b=40))
fig.show()

slider = widgets.IntSlider(value=5, min=0, max=10, description="Drag me")
label = widgets.Label("Move the slider…")
slider.observe(lambda change: setattr(label, "value", f"Slider is at {change['new']} ✅"), names="value")
display(slider, label)

## Next

All ticks, a chart and a working slider? Open **Chapter 1**
(`CHAPTER1_DATA_and_VISUALISATION.ipynb`).